In [178]:
# The City College of New York, City University of New York
# Written by Hasan Suca Kayman and Prof. M. Umit Uyar, September, 2025
# # Modified by Reaz Bhuiyan, March 2026
# NLP: Predicting financial stock movement based on tweets from Pres. Trump
# using ANNs

In [179]:
import re
import datetime
from datetime import date
import math
import numpy as np
import pandas as pd
from tensorflow import keras
from sklearn.preprocessing import MinMaxScaler
keras.utils.set_random_seed(42)

def print_weights(weights):
    # weights = model.get_weights();
    print('\n******* WEIGHTS OF ANN *******\n')
    for i in range(int(len(weights)/2)):
        print('Weights W%d:\n' %(i), weights[i*2])
        print('Bias b%d:\n' %(i), weights[(i*2)+1])

# Regular expression to remove punctuation and web links from tweets
# @\S+|https?://\S+ - matches either a substring which starts with @
# and contains non-whitespace characters \S+ OR a link(url) which
# starts with http(s)://
def clean_tweet(tweet):
    # delete all words starting with @
    temp = re.sub(r'@\w+','',tweet)
    # delete all words starting with #
    temp = re.sub(r'#\w+','',temp)
    # delete all words starting with url
    temp = re.sub(r'https?://\S+|www\.\S+','',temp)
    # delete all punctuation marks
    temp = re.sub(r'[^\w\s]','',temp)
    return temp

In [180]:
# read tweets
df = pd.read_csv('trumptwitter_2019_01_01_to_2020_03_11.csv')
df

,text,created_at
0,The Media should view this as a time of unity ...,3/11/20 22:04
1,I will be addressing the Nation this evening a...,3/11/20 20:44
2,I am fully prepared to use the full power of t...,3/11/20 19:18
3,....We have the greatest healthcare system exp...,3/11/20 19:17
4,I want to thank all of our Great Government of...,3/11/20 19:17
...,...,...
7717,RT @GOPChairwoman: Jobless claims fell last we...,1/1/19 14:50
7718,The Democrats much as I suspected have allocat...,1/1/19 14:32
7719,Happy New Year!,1/1/19 14:25
7720,HAPPY NEW YEAR TO EVERYONE INCLUDING THE HATER...,1/1/19 13:08


In [181]:
# clean-up the tweets:
df['clean_tweet'] = df['text'].apply(lambda X:clean_tweet(X.lower()))
df.iloc[7717,:]

,7717
text,RT @GOPChairwoman: Jobless claims fell last we...
created_at,1/1/19 14:50
clean_tweet,rt jobless claims fell last week to a 49year ...


In [182]:
# convert dates from string to date and eliminate hours:
df['created_at'] = pd.to_datetime(df['created_at']).dt.date
df.head()

/tmp/ipykernel_348/3970137872.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['created_at'] = pd.to_datetime(df['created_at']).dt.date


,text,created_at,clean_tweet
0,The Media should view this as a time of unity ...,2020-03-11,the media should view this as a time of unity ...
1,I will be addressing the Nation this evening a...,2020-03-11,i will be addressing the nation this evening a...
2,I am fully prepared to use the full power of t...,2020-03-11,i am fully prepared to use the full power of t...
3,....We have the greatest healthcare system exp...,2020-03-11,we have the greatest healthcare system experts...
4,I want to thank all of our Great Government of...,2020-03-11,i want to thank all of our great government of...


In [183]:
# define key words used in tweets that may be influential, which are
# selected by domain experts (e.g., in political science)
print('\n\n********** IDENTIFYING KEY WORDS **********\n\n')
key_words = ['Europe', 'China', 'tariff', 'Stock Market', 'economy', 'bank',
          'trade', 'jobs', 'money', 'dollar','currency','Xi','deal','growth']

# create a new column for number of keywords used in the tweet:
for key_word in key_words:
    df[key_word] = df.text.str.contains(key_word, case=False).astype(int)
df['noof_keywords'] = df[key_words].sum(axis=1)
# print noof_keyords column elements that are greater than 0 in the first 200 tweets:
for i in range(200):
    if (df.loc[i,'noof_keywords'] > 0):
        print (f"{i}: {df.loc[i,'noof_keywords']}")



********** IDENTIFYING KEY WORDS **********


2: 1
8: 1
29: 1
41: 1
60: 1
75: 1
89: 1
93: 1
105: 1
106: 1
112: 2
113: 1
151: 1
153: 2
154: 2
159: 1
173: 2
174: 1
175: 1
176: 1
181: 1
191: 1
196: 2


In [184]:
# combine tweet counts for the same day:
df_grouped = df.groupby('created_at')[key_words + ['noof_keywords']].sum()
df_grouped.reset_index(inplace=True)
df_grouped

,created_at,Europe,China,tariff,Stock Market,economy,bank,trade,jobs,money,dollar,currency,Xi,deal,growth,noof_keywords
0,2019-01-01,0,0,0,0,0,0,0,0,1,0,0,0,1,0,2
1,2019-01-02,0,0,0,0,0,0,1,1,0,1,0,1,1,0,5
2,2019-01-03,0,1,1,0,0,0,1,0,0,1,0,0,0,0,4
3,2019-01-04,0,1,0,0,0,0,0,1,0,0,0,0,0,1,3
4,2019-01-05,0,0,0,0,0,0,0,0,1,1,0,0,2,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
419,2020-03-07,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
420,2020-03-08,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1
421,2020-03-09,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1
422,2020-03-10,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1


In [185]:
#% Stock prices
filename = "S&P500_2019_01_01_to_2020_03_11.csv"
snp_df = pd.read_csv(filename)
snp_df['date'] = pd.to_datetime(snp_df['Date']).dt.date
snp_df['percent change'] = snp_df['Close'].pct_change()
snp_df['percent change'] = snp_df['percent change'].shift(-1)
snp_df

/tmp/ipykernel_348/3266677461.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  snp_df['date'] = pd.to_datetime(snp_df['Date']).dt.date


,Date,Open,High,Low,Close,Adj Close,Volume,date,percent change
0,1/2/19,2476.959961,2519.489990,2467.469971,2510.030029,2510.030029,3733160000,2019-01-02,-0.024757
1,1/3/19,2491.919922,2493.139893,2443.959961,2447.889893,2447.889893,3822860000,2019-01-03,0.034336
2,1/4/19,2474.330078,2538.070068,2474.330078,2531.939941,2531.939941,4213410000,2019-01-04,0.007010
3,1/7/19,2535.610107,2566.159912,2524.560059,2549.689941,2549.689941,4104710000,2019-01-07,0.009695
4,1/8/19,2568.110107,2579.820068,2547.560059,2574.409912,2574.409912,4083030000,2019-01-08,0.004098
...,...,...,...,...,...,...,...,...,...
295,3/5/20,3075.699951,3083.040039,2999.830078,3023.939941,3023.939941,5575550000,2020-03-05,-0.017054
296,3/6/20,2954.199951,2985.929932,2901.540039,2972.370117,2972.370117,6552140000,2020-03-06,-0.075970
297,3/9/20,2863.889893,2863.889893,2734.429932,2746.560059,2746.560059,8423050000,2020-03-09,0.049396
298,3/10/20,2813.479980,2882.590088,2734.000000,2882.229980,2882.229980,7635960000,2020-03-10,-0.048868


In [186]:
# combine df and snp_df based on date:
df_combo = pd.merge(df_grouped, snp_df, left_on='created_at', right_on='date', how='inner')
df_combo

,created_at,Europe,China,tariff,Stock Market,economy,bank,trade,jobs,money,...,noof_keywords,Date,Open,High,Low,Close,Adj Close,Volume,date,percent change
0,2019-01-02,0,0,0,0,0,0,1,1,0,...,5,1/2/19,2476.959961,2519.489990,2467.469971,2510.030029,2510.030029,3733160000,2019-01-02,-0.024757
1,2019-01-03,0,1,1,0,0,0,1,0,0,...,4,1/3/19,2491.919922,2493.139893,2443.959961,2447.889893,2447.889893,3822860000,2019-01-03,0.034336
2,2019-01-04,0,1,0,0,0,0,0,1,0,...,3,1/4/19,2474.330078,2538.070068,2474.330078,2531.939941,2531.939941,4213410000,2019-01-04,0.007010
3,2019-01-07,0,0,0,0,0,0,0,1,0,...,3,1/7/19,2535.610107,2566.159912,2524.560059,2549.689941,2549.689941,4104710000,2019-01-07,0.009695
4,2019-01-08,0,1,1,0,0,0,1,1,0,...,4,1/8/19,2568.110107,2579.820068,2547.560059,2574.409912,2574.409912,4083030000,2019-01-08,0.004098
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,2020-03-05,0,1,0,0,0,0,0,0,0,...,2,3/5/20,3075.699951,3083.040039,2999.830078,3023.939941,3023.939941,5575550000,2020-03-05,-0.017054
296,2020-03-06,0,0,0,0,0,0,0,1,0,...,1,3/6/20,2954.199951,2985.929932,2901.540039,2972.370117,2972.370117,6552140000,2020-03-06,-0.075970
297,2020-03-09,0,0,0,0,1,0,0,0,0,...,1,3/9/20,2863.889893,2863.889893,2734.429932,2746.560059,2746.560059,8423050000,2020-03-09,0.049396
298,2020-03-10,0,0,0,0,0,0,0,0,0,...,1,3/10/20,2813.479980,2882.590088,2734.000000,2882.229980,2882.229980,7635960000,2020-03-10,-0.048868


In [187]:
features = [ 'Open', 'Volume','noof_keywords',] + key_words
features = features + ['percent change']
df = df_combo[features]
df

,Open,Volume,noof_keywords,Europe,China,tariff,Stock Market,economy,bank,trade,jobs,money,dollar,currency,Xi,deal,growth,percent change
0,2476.959961,3733160000,5,0,0,0,0,0,0,1,1,0,1,0,1,1,0,-0.024757
1,2491.919922,3822860000,4,0,1,1,0,0,0,1,0,0,1,0,0,0,0,0.034336
2,2474.330078,4213410000,3,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0.007010
3,2535.610107,4104710000,3,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0.009695
4,2568.110107,4083030000,4,0,1,1,0,0,0,1,1,0,0,0,0,0,0,0.004098
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,3075.699951,5575550000,2,0,1,0,0,0,0,0,0,0,0,0,1,0,0,-0.017054
296,2954.199951,6552140000,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,-0.075970
297,2863.889893,8423050000,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0.049396
298,2813.479980,7635960000,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,-0.048868


In [188]:
# Normalize the volume and open:
df = df.dropna()
open_min = df['Open'].min()
open_max = df['Open'].max()
vol_min = df['Volume'].min()
vol_max = df['Volume'].max()
keywds_min = df['noof_keywords'].min()
keywds_max = df['noof_keywords'].max()

df.loc[:,'Open'] = (df['Open'] - open_min)/(open_max - open_min)
df.loc[:,'Volume'] = (df['Volume'] - vol_min)/(vol_max - vol_min)
df.loc[:,'noof_keywords'] = (df['noof_keywords'] - keywds_min)/(keywds_max - keywds_min)
df

/tmp/ipykernel_348/1245740493.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.335285   0.34762794 0.40136859 0.3864112  0.38342798 0.37922422
 0.33134131 0.29418726 0.32583033 0.31315439 0.35325726 0.34066663
 0.37017686 0.35934754 0.28058112 0.29401663 0.34641979 0.31872453
 0.30377953 0.35381317 0.49827378 0.3388778  0.28391523 0.31151692
 0.29944367 0.38569292 0.32003451 0.28420832 0.34830357 0.32669998
 0.34953236 0.32265446 0.30784018 0.34936035 0.31141784 0.29326807
 0.34508505 0.32324753 0.33995935 0.4266214  0.3681885  0.3609685
 0.31499276 0.34263847 0.35891134 0.29262409 0.33746875 0.29139943
 0.3398245  0.29903637 0.64207939 0.31038307 0.31974417 0.34051939
 0.30964139 0.40464216 0.2862187  0.27100949 0.28571645 0.25616494
 0.33632252 0.30330618 0.26837441 0.31011475 0.23648915 0.25460315
 0.24183501 0.23549842 0.24298399 0.2259433  0.32913829 0.24655478
 0.28974545 0.31727833 0.30414417 0.234118

,Open,Volume,noof_keywords,Europe,China,tariff,Stock Market,economy,bank,trade,jobs,money,dollar,currency,Xi,deal,growth,percent change
0,0.002902,0.335285,0.087719,0,0,0,0,0,0,1,1,0,1,0,1,1,0,-0.024757
1,0.019412,0.347628,0.070175,0,1,1,0,0,0,1,0,0,1,0,0,0,0,0.034336
2,0.000000,0.401369,0.052632,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0.007010
3,0.067629,0.386411,0.052632,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0.009695
4,0.103496,0.383428,0.070175,0,1,1,0,0,0,1,1,0,0,0,0,0,0,0.004098
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
294,0.630623,0.514487,0.087719,0,0,0,0,2,0,0,0,1,1,0,0,1,0,-0.033922
295,0.663676,0.588802,0.035088,0,1,0,0,0,0,0,0,0,0,0,1,0,0,-0.017054
296,0.529588,0.723184,0.017544,0,0,0,0,0,0,0,1,0,0,0,0,0,0,-0.075970
297,0.429921,0.980626,0.017544,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0.049396


In [189]:
# Prepare input and output features for ANN
X = np.array(df.iloc[:,:-1])
Y = np.array(df.loc[:,'percent change'])
X.shape, Y.shape

((299, 17), (299,))

In [190]:
X

array([[0.00290236, 0.335285  , 0.0877193 , ..., 1.        , 1.        ,
        0.        ],
       [0.01941227, 0.34762794, 0.07017544, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.40136859, 0.05263158, ..., 0.        , 0.        ,
        1.        ],
       ...,
       [0.52958763, 0.72318368, 0.01754386, ..., 0.        , 0.        ,
        0.        ],
       [0.42992084, 0.98062557, 0.01754386, ..., 0.        , 0.        ,
        0.        ],
       [0.37428812, 0.87232002, 0.01754386, ..., 0.        , 0.        ,
        0.        ]])

In [191]:
Y

array([-2.47567301e-02,  3.43357143e-02,  7.01043485e-03,  9.69528514e-03,
        4.09804552e-03,  4.51841892e-03, -1.46297914e-04, -5.25752542e-03,
        1.07216889e-02,  2.22198555e-03,  7.59140027e-03,  1.31830530e-02,
       -1.41573063e-02,  2.20291284e-03,  1.37572557e-03,  8.48869420e-03,
       -7.84682745e-03, -1.45624671e-03,  1.55492610e-02,  8.59739601e-03,
        8.98609856e-04,  6.77623666e-03,  4.70842038e-03, -2.22443807e-03,
       -9.35713993e-03,  6.76201093e-04,  7.09103093e-04,  1.28902245e-02,
        3.02399473e-03, -2.65164162e-03,  1.08787529e-02,  1.49874328e-03,
        1.77711061e-03, -3.52643666e-03,  6.41110237e-03,  1.23186237e-03,
       -7.90457069e-04, -5.44049198e-04, -2.82550847e-03,  6.89532053e-03,
       -3.88055820e-03, -1.13153309e-03, -6.52409859e-03, -8.12571737e-03,
       -2.13168911e-03,  1.46660421e-02,  2.95331831e-03,  6.94958369e-03,
       -8.68022593e-04,  4.98490290e-03,  3.70594692e-03, -1.30561539e-04,
       -2.94435364e-03,  

In [192]:
# create a model for the ANN:
model = keras.Sequential()
alpha = 0.01
# add a hidden layer that takes in 17 input features
model.add(keras.layers.Input(shape=(17,)))
# add a hidden layer with 4 neurons

model.add(keras.layers.Dense(10, activation='sigmoid'))
model.add(keras.layers.Dense(4, activation='sigmoid',))
# add another hidden layer with 8 neurons
model.add(keras.layers.Dense(8, activation='sigmoid'))
# add an output layer with one output (the slope of the cognitive decline)
# activation function default to linear
model.add(keras.layers.Dense(16, activation='sigmoid'))
model.add(keras.layers.Dense(8, activation='sigmoid'))

model.add(keras.layers.Dense(1, activation='tanh'))

# set optimizer to Adam which is a gradient descent method that updates
# the weights to minimize the loss
# set loss to mean_absolute_error which does the average absolute
# difference between actual output and estimated output
# set learning rate to 0.01, learning rate is the rate at which model
# paremeters are updated)
# metrics is utilized to see how the model is performing each epoch
model.compile(optimizer=keras.optimizers.SGD(learning_rate=alpha),
              loss='mean_absolute_error',
              metrics=[keras.metrics.RootMeanSquaredError()]
              )
model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_49 (Dense)                │ (None, 10)             │           180 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_50 (Dense)                │ (None, 4)              │            44 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_51 (Dense)                │ (None, 8)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_52 (Dense)                │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_53 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_54 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 553 (2.16 KB)

 Trainable params: 553 (2.16 KB)

 Non-trainable params: 0 (0.00 B)

In [193]:
# print the randomly initialized weights:
weights = model.get_weights()
print_weights(weights)


******* WEIGHTS OF ANN *******

Weights W0:
 [[ 0.3377039   0.15332767 -0.13320878  0.4242542  -0.3978078   0.14362028
   0.44359156 -0.31309545  0.02242875  0.12328133]
 [-0.38992858 -0.05057463  0.4393808  -0.02796268  0.00277787  0.25047234
  -0.4536515  -0.29002607  0.01342708 -0.4087787 ]
 [-0.4359125   0.32806602  0.39166918 -0.22387867  0.33896992 -0.35122037
  -0.03555766  0.05913475  0.06287768 -0.28268993]
 [-0.27370435  0.4611881  -0.36779645 -0.05316189 -0.22147663 -0.35730457
   0.21980932 -0.29949212  0.17092726  0.1418164 ]
 [-0.43121183  0.05999646 -0.15639156 -0.10668188  0.4285269  -0.08448127
   0.02967039 -0.23068678 -0.35561138  0.20348647]
 [ 0.1097559   0.00750056 -0.4452303  -0.35857022  0.27472708  0.3452371
  -0.06425259 -0.17558962 -0.44023675 -0.4349247 ]
 [ 0.24424765  0.30982307 -0.11692593  0.20700082 -0.1733909   0.26296756
  -0.17415372 -0.07216227  0.33449063  0.30866495]
 [ 0.17585364  0.37419376  0.33262125 -0.25871468  0.12772372 -0.38669014
   0.4

In [194]:
## train the ANN model using 120 iterations
model.fit(X, Y, epochs=200, verbose = 0)
print('\n\n********** ANN training complete **********\n\n')



********** ANN training complete **********




In [195]:
# print the weights of the trained ANN:
weights = model.get_weights()
print_weights(weights)


******* WEIGHTS OF ANN *******

Weights W0:
 [[ 0.33779272  0.15348779 -0.1332844   0.4243033  -0.39783046  0.14357197
   0.44361386 -0.3130934   0.02250453  0.12324684]
 [-0.38986462 -0.05046521  0.43932617 -0.02792721  0.00276226  0.25043815
  -0.45364332 -0.29002452  0.01348129 -0.40880078]
 [-0.4359202   0.3280938   0.39166984 -0.22387867  0.33897164 -0.35121676
  -0.03555498  0.05913286  0.06286589 -0.28269085]
 [-0.2737073   0.46116427 -0.3677833  -0.05315543 -0.22148176 -0.35729113
   0.21981107 -0.29949874  0.17093915  0.14181423]
 [-0.43132883  0.06042657 -0.15630509 -0.10668392  0.4285446  -0.08444518
   0.02967168 -0.23070617 -0.35580325  0.2035127 ]
 [ 0.10963058  0.00763797 -0.4452024  -0.3586496   0.2747662   0.3452885
  -0.06424001 -0.17561074 -0.44040105 -0.43493277]
 [ 0.24427608  0.30986783 -0.11694869  0.20699477 -0.17339869  0.26296622
  -0.1741471  -0.07215911  0.3345045   0.30865708]
 [ 0.17578445  0.3743889   0.3326506  -0.25870672  0.12777655 -0.38661984
   0.4

In [196]:
# Test on the ANN using the last 10 days:
predictions = model.predict(X[-10:])
for i in range (10):
    if ((Y[-10+i] > 0 and predictions[i] > 0) or (Y[-10+i] <= 0 and predictions[i] <= 0)):
        print(f'Day: {10-i}:\tReal Movement: {Y[-10+i]}\tPrediction: {predictions[i]}\tMATCH')
    else:
        print(f'Day: {10-i}:\tReal Movement: {Y[-10+i]}\tPrediction: {predictions[i]}\tNO MATCH')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
Day: 10:	Real Movement: -0.04416324263826643	Prediction: [-0.01162697]	MATCH
Day: 9:	Real Movement: -0.00823834042273175	Prediction: [-0.011566]	MATCH
Day: 8:	Real Movement: 0.04603922874232036	Prediction: [-0.01161109]	NO MATCH
Day: 7:	Real Movement: -0.02810789603432695	Prediction: [-0.01206578]	MATCH
Day: 6:	Real Movement: 0.04220259077712596	Prediction: [-0.01206895]	NO MATCH
Day: 5:	Real Movement: -0.033922077118805904	Prediction: [-0.01225312]	MATCH
Day: 4:	Real Movement: -0.01705385193032183	Prediction: [-0.01128427]	MATCH
Day: 3:	Real Movement: -0.07596969728248681	Prediction: [-0.01174338]	MATCH
Day: 2:	Real Movement: 0.04939630595567479	Prediction: [-0.01166368]	NO MATCH
Day: 1:	Real Movement: -0.048868444911533415	Prediction: [-0.01164908]	MATCH


In [ ]:
## ask user if they would like to save the ANN model
choice = ''
while choice not in ['y','n']:
    choice = input('\n\nWould you like to save the ANN model? (y/n): \n')
    if choice == 'y':
        save_name = input('\n\nEnter a name for the save file: \n')
        ## if file name does not end with '.h5', add '.h5' to the file name
        if save_name[-3:] != '.h5':
            save_name += '.h5'
        model.save(save_name)
        print('\n\n')
        print('***** ANN MODEL SUCCESSFULLY SAVED AS '+save_name+' *****')
    elif choice == 'n':
        pass
    else:
        print("Invalid input: Input must be 'y' or 'n'")



Would you like to save the ANN model? (y/n): 
y
